Cell 1: Setup Model & Pembuatan Fungsi Prediksi

In [3]:
import pandas as pd
import json
import os
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB


# Persiapan Data & Model (Retrieval)
# Load data train
df_train = pd.read_csv("../data/processed/cases.csv")
df_train['teks_pencarian'] = df_train['ringkasan_fakta'].fillna('') + " " + df_train['text_full'].fillna('')

# Latih ulang model TF-IDF & Naive Bayes
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df_train['teks_pencarian'])
nb_model = MultinomialNB()
nb_model.fit(tfidf_matrix, df_train['case_id'])

# Fungsi bawaan dari Tahap 3 (Hanya mengembalikan list case_id)
def retrieve(query: str, k: int = 5):
    query_vec = vectorizer.transform([query])
    probabilitas = nb_model.predict_proba(query_vec)[0]
    top_k_indices = probabilitas.argsort()[-k:][::-1]
    return [str(nb_model.classes_[idx]) for idx in top_k_indices]

# i. Ekstrak Solusi
# 2. Simpan di struktur: {case_id: solusi_text}
case_solutions = dict(zip(df_train['case_id'], df_train['amar_putusan']))


# ii & iii. Implementasi Fungsi Prediksi (Majority Vote)
def predict_outcome(query: str) -> str:
    # Ambil top-k kasus yang mirip
    top_k = retrieve(query, k=5)
    
    # Ambil list teks solusi dari top-k kasus tersebut
    solutions = [case_solutions[c] for c in top_k]
    
    # Terapkan algoritma prediksi: Majority Vote
    # Menghitung teks solusi mana yang paling sering muncul
    vote_count = Counter(solutions)
    predicted_solution = vote_count.most_common(1)[0][0]
    
    # Mengembalikan 2 nilai agar bisa dimasukkan ke tabel CSV nanti
    return predicted_solution, top_k

print("Struktur case_solutions dan fungsi predict_outcome() siap digunakan!")

Struktur case_solutions dan fungsi predict_outcome() siap digunakan!


Cell 2: Demo Manual & Penyimpanan CSV (Poin iv & v)

Cell ini akan membandingkan hasil prediksi dengan jawaban aslinya, lalu mencetak tabel CSV dengan struktur kolom yang persis seperti tabel di instruksi: query id, predicted solution, top 5 case ids.

In [4]:

# iv. Demo Manual
print("Memulai Demo Manual: Membandingkan Prediksi vs Putusan Sebenarnya\n")

# Siapkan 5 contoh kasus baru (dari file queries.json)
path_queries = "../data/eval/queries.json"
with open(path_queries, "r", encoding="utf-8") as f:
    queries = json.load(f)

hasil_prediksi = []

for q in queries:
    # Jalankan predict_outcome()
    pred_sol, top_5_ids = predict_outcome(q['query_text'])
    
    # Ambil putusan sebenarnya (Ground Truth) untuk dibandingkan
    ground_truth_id = q['ground_truth_case_id']
    putusan_sebenarnya = case_solutions.get(ground_truth_id, "TIDAK DITEMUKAN")
    
    # Cetak ke layar untuk demo manual
    print(f"[*] Query ID          : {q['query_id']}")
    print(f"    Top 5 Case IDs    : {top_5_ids}")
    print(f"    Predicted Solution: {pred_sol[:150]}...") # Dipotong agar rapi di layar
    print(f"    Putusan Sebenarnya: {putusan_sebenarnya[:150]}...")
    print("-" * 70)
    
    
    # v. Output: Siapkan baris untuk tabel CSV
    hasil_prediksi.append({
        "query id": q['query_id'],
        "predicted solution": pred_sol,
        "top 5 case ids": ", ".join(top_5_ids)
    })

# Simpan ke File /data/results/predictions.csv
folder_results = "../data/results"
os.makedirs(folder_results, exist_ok=True)
path_pred_csv = os.path.join(folder_results, "predictions.csv")

df_predictions = pd.DataFrame(hasil_prediksi)
df_predictions.to_csv(path_pred_csv, index=False)

print(f"Selesai! File output berhasil disimpan di: {path_pred_csv}")

Memulai Demo Manual: Membandingkan Prediksi vs Putusan Sebenarnya

[*] Query ID          : query_case_028
    Top 5 Case IDs    : ['case_028', 'case_019', 'case_027', 'case_024', 'case_020']
    Predicted Solution: disclaimer kepaniteraan mahkamah agung republik indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen ...
    Putusan Sebenarnya: disclaimer kepaniteraan mahkamah agung republik indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen ...
----------------------------------------------------------------------
[*] Query ID          : query_case_016
    Top 5 Case IDs    : ['case_016', 'case_019', 'case_027', 'case_024', 'case_022']
    Predicted Solution: perkara aquo berkenan memutuskan sebagai berikut: 1. menerima dan mengabulkan gugatan penggugat untuk seluruhnya ; 2. mnyatakan bahwa merek "htc" mili...
    Putusan Sebenarnya: perkara aquo berkenan memutuskan sebagai berikut: 1. men